# Comprobación integridad referencial archivos movieLens

Este cuaderno comprueba y garantiza la integridad referencial entre todos los datasets ya limpiados de forma individual (movies, ratings, tags, links y TMDB), y elimina los registros huérfanos que pudieran quedar tras cruzar la información entre ellos. Aviso: este archivo se debe cargar después de todos los cuadernillos clean_...

**Entrada**: movies_clean.parquet, ratings_clean.parquet, tags_clean.parquet, links_clean.parquet, TMDB_clean.parquet \
**Objetivo**: verificar que los identificadores compartidos entre datasets son consistentes entre sí, y eliminar los registros que queden huérfanos tras el cruce \
**Salida**: links_integrity.parquet, ratings_integrity.parquet, tags_integrity.parquet, movies_integrity.parquet, TMDB_integrity.parquet

El proceso se organiza en tres partes: primero se valida la coherencia de los identificadores movieId y userId entre datasets; después se eliminan las películas sin datos válidos de TMDB (por fallo en la petición o por tmdbId nulo); y finalmente se consolida el conjunto con TMDB para quedarse únicamente con las películas presentes en todas las fuentes, guardando el resultado final.

## Validación de integridad referencial

Se comprueba que la columna movieId de tags.parquet y ratings.parquet exista en movies.parquet, y que la columna userId de tags.parquet esté contenida en la columna userId de ratings.parquet.

In [143]:
import pandas as pd

Se cargan las versiones ya limpiadas de todos los datasets, generadas en los cuadernos de limpieza individuales.

In [144]:
movies = pd.read_parquet("../data/02_processed/movies_clean.parquet")
tags = pd.read_parquet("../data/02_processed/tags_clean.parquet")
ratings = pd.read_parquet("../data/02_processed/ratings_clean.parquet")
links = pd.read_parquet("../data/02_processed/links_clean.parquet")

Se seleccionan las columnas userId y movieId de los respectivos datasets, para poder comparar con `isin()` qué identificadores de un dataset no aparecen en otro.

In [145]:
movieId_movies = movies['movieId']
movieId_tags = tags['movieId']
movieId_ratings = ratings['movieId']
movieId_links = links['movieId']

userId_tags = tags['userId']
userId_ratings = ratings['userId']

In [146]:
ratings[~movieId_ratings.isin(movieId_movies)]
# cumple 

,userId,movieId,rating,timestamp


In [147]:
links[~movieId_links.isin(movieId_movies)]
# cumple 
# se conoce de pasos anteriores la existencia de registros en movies que no están en links

,movieId,tmdbId


In [148]:
tags[~movieId_tags.isin(movieId_movies)]
# cumple

,userId,movieId,tag,timestamp


Los tres resultados anteriores están vacíos, por lo que todos los movieId presentes en ratings, links y tags existen en movies. En el caso de links, ya se sabía por el cuaderno de limpieza de links que existen películas en movies sin link válido asociado; esa dirección se trata más adelante, en la limpieza basada en TMDB.

In [149]:
tags[~movieId_tags.isin(movieId_ratings)]
# hay películas que tienen calificación en tags pero no tienen ratings asociado. No influye

,userId,movieId,tag,timestamp
560,288,7020,notable nudity,2006-06-22 14:54:37
584,318,30892,animation,2009-04-19 16:29:50
585,318,30892,documentary,2009-04-19 16:29:11
586,318,30892,henry darger,2009-04-19 16:29:20
1275,474,1076,governess,2006-01-17 18:09:54
1705,474,2939,in netflix queue,2006-01-14 01:29:00
1778,474,3338,in netflix queue,2006-01-14 01:21:20
1798,474,3456,in netflix queue,2006-01-14 01:17:43
1887,474,4194,in netflix queue,2006-01-14 01:19:35
2031,474,5721,in netflix queue,2006-01-14 01:09:36


Se compara ahora directamente el movieId de tags con el de ratings. No se trata de identificadores huérfanos respecto a movies, sino de películas que tienen etiquetas pero no calificación, o viceversa, ya que etiquetar y calificar son eventos independientes.

In [150]:
ratings[~movieId_ratings.isin(movieId_tags)]
# Existen películas que tienen calificaciones en ratings que no tienen calificaciones en tags. No influye

,userId,movieId,rating,timestamp
2,1,6,4.0,2000-07-30 18:37:04
5,1,70,3.0,2000-07-30 18:40:00
8,1,151,5.0,2000-07-30 19:07:21
9,1,157,5.0,2000-07-30 19:08:20
10,1,163,5.0,2000-07-30 19:00:50
...,...,...,...,...
100828,610,163981,3.5,2017-05-03 22:22:35
100830,610,166528,4.0,2017-05-04 06:29:25
100831,610,166534,4.0,2017-05-03 21:53:22
100833,610,168250,5.0,2017-05-08 19:50:47


No se considera que sea un problema de identidad referencial, ya que presentan eventos diferentes y se usarán para distintos tipos de recomendaciones.

In [151]:
tags[~userId_tags.isin(userId_ratings)]
# todos los usuarios que tienen calificaiones en tags tambien tienen calificaciones en ratings

,userId,movieId,tag,timestamp


El resultado está vacío, por lo que todos los usuarios que tienen etiquetas en tags también tienen calificaciones en ratings.

In [152]:
ratings[~userId_ratings.isin(userId_tags)]
# existen usuarios que solo tienen calificaciones en ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04
3,1,47,5.0,2000-07-30 19:03:35
4,1,50,5.0,2000-07-30 18:48:51
...,...,...,...,...
99529,609,892,3.0,1996-11-05 19:11:20
99530,609,1056,3.0,1996-11-05 19:11:20
99531,609,1059,3.0,1996-11-05 19:10:54
99532,609,1150,4.0,1996-11-05 19:10:54


Existen usuarios con calificaciones en ratings que nunca han puesto una etiqueta. Es un comportamiento esperado, ya que no todos los usuarios que valoran películas también las etiquetan, por lo que no se considera un problema de integridad y no requiere limpieza.

Conclusión: todos los identificadores de película (movieId) de ratings y tags deben existir en movies_clean.parquet. En caso de que no se cumpla, se eliminarán las calificaciones o etiquetas asociadas a las películas inexistentes.

El archivo requestTMDB.log contiene los ids que habían fallado al realizar la petición.  Estos archivos no cuentan con datos de  películas en formato JSON, para que no haya problemas de integridad se eliminarán las filas asociadas en los todos archivos.

movieId de películas que fallaron 

In [153]:
logs = pd.read_csv("../logs/requestTMDB.log", header = None, sep="|", names = ['timestamp','levelname','tmdbId', 'message'])
logs.head()

,timestamp,levelname,tmdbId,message
0,"2026-06-06 18:42:39,035",WARNING,tmdbId = 876,HTTP_Status = 404
1,"2026-06-06 18:44:47,553",WARNING,tmdbId = 2670,HTTP_Status = 404
2,"2026-06-06 18:55:27,134",WARNING,tmdbId = 7096,HTTP_Status = 404
3,"2026-06-06 18:56:18,174",WARNING,tmdbId = 8677,HTTP_Status = 404
4,"2026-06-06 18:58:14,545",WARNING,tmdbId = 9795,HTTP_Status = 404


In [154]:
# se extraen  lod id de la columna tmdbId
tmdbId_logs = logs['tmdbId'].str.extract("([0-9]+)").astype('Int64')

#se sacan los movieId asociados a los tmdbId
movieIds_logs = links.loc[links['tmdbId'].isin(tmdbId_logs[0]),'movieId'].tolist()
movieIds_logs


[4207,
 4568,
 5069,
 5209,
 7646,
 7669,
 7762,
 7841,
 7842,
 26453,
 26614,
 26649,
 26693,
 26761,
 26849,
 26887,
 27002,
 27036,
 27251,
 27611,
 27708,
 27751,
 31193,
 38198,
 49917,
 51024,
 52281,
 53883,
 55207,
 57772,
 61406,
 62336,
 62970,
 63433,
 64167,
 65135,
 65359,
 66544,
 66934,
 69524,
 69849,
 70521,
 71160,
 72982,
 77177,
 84847,
 85780,
 86237,
 86487,
 90647,
 90863,
 90945,
 92475,
 93006,
 93008,
 93040,
 93988,
 95717,
 95738,
 96471,
 96518,
 96520,
 99532,
 99764,
 100044,
 100553,
 105250,
 106642,
 107780,
 108727,
 115969,
 119218,
 121035,
 122260,
 122926,
 126430,
 127180,
 127390,
 130842,
 131724,
 137859,
 139130,
 140481,
 140737,
 141816,
 142115,
 147250,
 147328,
 147330,
 148675,
 150548,
 151763,
 152284,
 159817,
 163809,
 167570,
 170355,
 170705,
 171011,
 171495,
 171749,
 172497,
 172909,
 173535,
 173873,
 174053,
 174403,
 175693,
 176329,
 179135,
 180263,
 184257,
 185135]

In [155]:
# se eliminan de todos los archivos los registros asociados a estas peliculas
links = links[~links['movieId'].isin(tmdbId_logs[0])]
ratings = ratings[~ratings['movieId'].isin(tmdbId_logs[0])]
tags = tags[~tags['movieId'].isin(tmdbId_logs[0])]
movies = movies[~movies['movieId'].isin(tmdbId_logs[0])]

In [156]:
ratings['movieId'].unique()

array([     1,      3,      6, ..., 160836, 163937, 163981], shape=(9719,))

se eliminan registros que no tienen tmdbId en links de todas las tablas que hacen referencia

In [157]:
movieId_elim = links[links['tmdbId'].isna()]['movieId']
movieId_elim.tolist()

[791, 1107, 2851, 4051, 26587, 32600, 40697, 79299]

In [158]:
links.dropna(subset=['tmdbId'], inplace=True)
ratings = ratings[~ratings['movieId'].isin(movieId_elim)]
tags = tags[~tags['movieId'].isin(movieId_elim)]
movies = movies[~movies['movieId'].isin(movieId_elim)]

In [159]:
links[links['tmdbId'].isna()]

,movieId,tmdbId


In [160]:
TMDB = pd.read_parquet("../data/02_processed/TMDB_clean.parquet")


se comprueba los registros de todas las tablas solo tienen peliculas de TMDB

In [161]:
TMDB

,id,title,genres,popularity,overview,tagline,vote_average,vote_count,runtime,budget,revenue,release_date
0,2,Ariel,"[Comedy, Drama, Romance, Crime]",1.3807,A Finnish man goes to the city to find a job a...,,7.121,375,73,0,0,1988-10-21
1,5,Four Rooms,[Comedy],3.6807,It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,5.904,2847,98,4000000,4257354,1995-12-09
2,6,Judgment Night,"[Action, Crime, Thriller]",1.8049,"Four young friends, while taking a shortcut en...",Don't move. Don't whisper. Don't even breathe.,6.463,377,109,21000000,12136938,1993-10-15
3,11,Star Wars,"[Adventure, Action, Science Fiction]",27.0355,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",8.205,22361,121,11000000,775398007,1977-05-25
4,12,Finding Nemo,"[Animation, Family, Adventure]",19.5501,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,7.819,20567,100,94000000,940335536,2003-05-30
...,...,...,...,...,...,...,...,...,...,...,...,...
9615,497520,Tom Segura: Disgraceful,[Comedy],0.2180,Tom Segura gives voice to the sordid thoughts ...,,7.100,81,74,0,0,2018-01-12
9616,500475,SuperFly,"[Action, Crime]",2.7139,Career criminal Youngblood Priest wants out of...,Redefine the hussle,6.706,406,107,16000000,20545116,2018-06-13
9617,502616,Fred Armisen: Standup for Drummers,"[Music, Comedy]",0.2086,"For an audience of drummers, comedian Fred Arm...","I don't want to work, I just want to bang on t...",6.300,33,65,0,0,2018-02-06
9618,503475,Wallace & Gromit: The Best of Aardman Animation,"[Family, Animation, Comedy]",0.6346,Anthology of Aardman Animation short films rel...,The Best of Aardman Animation,7.600,5,75,0,0,1996-06-14


In [162]:
links

,movieId,tmdbId
0,1,862
1,2,8844
2,3,15602
3,4,31357
4,5,11862
...,...,...
9737,193581,432131
9738,193583,445030
9739,193585,479308
9740,193587,483455


In [163]:
links

,movieId,tmdbId
0,1,862
1,2,8844
2,3,15602
3,4,31357
4,5,11862
...,...,...
9737,193581,432131
9738,193583,445030
9739,193585,479308
9740,193587,483455


In [164]:
linkTMDB = TMDB.merge(links, left_on='id', right_on='tmdbId', how='inner');linkTMDB

,id,title,genres,popularity,overview,tagline,vote_average,vote_count,runtime,budget,revenue,release_date,movieId,tmdbId
0,2,Ariel,"[Comedy, Drama, Romance, Crime]",1.3807,A Finnish man goes to the city to find a job a...,,7.121,375,73,0,0,1988-10-21,4470,2
1,5,Four Rooms,[Comedy],3.6807,It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,5.904,2847,98,4000000,4257354,1995-12-09,18,5
2,6,Judgment Night,"[Action, Crime, Thriller]",1.8049,"Four young friends, while taking a shortcut en...",Don't move. Don't whisper. Don't even breathe.,6.463,377,109,21000000,12136938,1993-10-15,479,6
3,11,Star Wars,"[Adventure, Action, Science Fiction]",27.0355,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",8.205,22361,121,11000000,775398007,1977-05-25,260,11
4,12,Finding Nemo,"[Animation, Family, Adventure]",19.5501,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,7.819,20567,100,94000000,940335536,2003-05-30,6377,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9611,497520,Tom Segura: Disgraceful,[Comedy],0.2180,Tom Segura gives voice to the sordid thoughts ...,,7.100,81,74,0,0,2018-01-12,183959,497520
9612,500475,SuperFly,"[Action, Crime]",2.7139,Career criminal Youngblood Priest wants out of...,Redefine the hussle,6.706,406,107,16000000,20545116,2018-06-13,189381,500475
9613,502616,Fred Armisen: Standup for Drummers,"[Music, Comedy]",0.2086,"For an audience of drummers, comedian Fred Arm...","I don't want to work, I just want to bang on t...",6.300,33,65,0,0,2018-02-06,184791,502616
9614,503475,Wallace & Gromit: The Best of Aardman Animation,"[Family, Animation, Comedy]",0.6346,Anthology of Aardman Animation short films rel...,The Best of Aardman Animation,7.600,5,75,0,0,1996-06-14,720,503475


In [165]:
tmdbId_validos = linkTMDB['id']
movieId_validos = linkTMDB['movieId']

In [166]:
links = links[links['movieId'].isin(movieId_validos)]
ratings = ratings[ratings['movieId'].isin(movieId_validos)]
tags = tags[tags['movieId'].isin(movieId_validos)]
movies = movies[movies['movieId'].isin(movieId_validos)]
TMDB = TMDB[TMDB['id'].isin(tmdbId_validos)]

Se guardan nuevas versiones de todos los datasets

In [167]:
links.to_parquet('../data/02_processed/links_integrity.parquet', index=False)
ratings.to_parquet('../data/02_processed/ratings_integrity.parquet', index=False)
tags.to_parquet('../data/02_processed/tags_integrity.parquet', index=False)
movies.to_parquet('../data/02_processed/movies_integrity.parquet', index=False)
TMDB.to_parquet('../data/02_processed/TMDB_integrity.parquet', index=False)